Compare FSTs between Cladocopium population in different Acropora hosts \
Jaelyn Bos, 2026

In [2]:
#Make sure R can find packages installed through Conda
.libPaths('/hb/home/jbos/.conda/envs/vcfR')
.libPaths("/hb/home/jbos/.conda/envs/vcfR/lib/R/library")

In [3]:
#Load packages
library(tidyverse)
library(vcfR)
library(hierfstat)
library(adegenet)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   4.0.0     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

   *****       ***   vcfR   ***       *****
   This is vcfR 1.15.0 
     browseVignettes('vcfR') # Documentation
     citation('vcfR') # Citation
   *****       *****      *****       *****


Loading required package: ade4


   /// adegenet 2.1.11 is loaded ////////////

   > overview: '?adegenet'
   > tutorials/doc/questions: 'adegenetWeb()' 
   > bug reports/feature requests: adegenetIssues()




Attaching package: ‘adegenet’


The following objects are masked from ‘pa

In [4]:
#Import Cladocopium SNP data in vcf form and convert to genind
vcf<-read.vcfR("vcfs/pruned_snps_cladocopium.vcf")
gi <- vcfR2genind(vcf)

Scanning file to determine attributes.
File attributes:
  meta lines: 699
  header_line: 700
  variant count: 873
  column count: 319
Meta line 699 read in.
All meta lines processed.
gt matrix initialized.
Character matrix gt created.
  Character matrix gt rows: 873
  Character matrix gt cols: 319
  skip: 0
  nrows: 873
  row_num: 0
Processed variant: 873
All variants processed


In [5]:
#Import names of individuals samples assigned to each Acropora cryptic taxon
taxa1<-read.table('other_data/taxa1_dfa.txt',header=FALSE)$V1
taxa2<-read.table('other_data/taxa2_dfa.txt',header=FALSE)$V1
taxa3<-read.table('other_data/taxa3_dfa.txt',header=FALSE)$V1
taxa4<-read.table('other_data/taxa4_dfa.txt',header=FALSE)$V1

In [6]:
#Make a list of individual corals in the Cladocopium dataset
inds<-as.data.frame(matrix(nrow=nrow(gi@tab),ncol=1))
inds$ind<-rownames(gi@tab)
colnames(inds)<-c('taxa','ind')

In [7]:
#Replace names with number of appropriate taxon
inds$taxa<-0
inds[inds$ind %in% taxa1,1]<-1
inds[inds$ind %in% taxa2,1]<-2
inds[inds$ind %in% taxa3,1]<-3
inds[inds$ind %in% taxa4,1]<-4

In [8]:
#Set Acropora cyptic taxon as 'population' variable within the Cladocopium genind
gi@pop<-as.factor(inds$taxa)

In [9]:
#Convert to hierfstat
hstat1<-genind2hierfstat(gi,pop=gi@pop)

In [10]:
#Calculate pairwise FSTs based on cryptic Acropora taxa
fst_taxa<-pairwise.WCfst(hstat1)

In [11]:
#Print FSTs
fst_taxa

,0,1,2,3,4
0,NA,-0.0025802948,-0.0019841454,-0.004842227,-0.0010510081
1,-0.002580295,NA,0.0006921428,0.004740272,0.0006911759
2,-0.001984145,0.0006921428,NA,0.003350502,0.0040124343
3,-0.004842227,0.0047402717,0.0033505024,NA,0.0018680093
4,-0.001051008,0.0006911759,0.0040124343,0.001868009,NA


In [12]:
#Check number of individuals per taxon with Cladocopium data
table(inds$taxa)


  0   1   2   3   4 
 24 113  71  56  46 

In [13]:
#Read in bootstrapped null FST distributions of Cladocopium pops between different Acropora taxa
boot12<-ecdf(read.csv('/scratch/jbos/cladocopium/cladocopium12_fst_boot.csv')[,2])
boot13<-ecdf(read.csv('/scratch/jbos/cladocopium/cladocopium13_fst_boot.csv')[,2])
boot14<-ecdf(read.csv('/scratch/jbos/cladocopium/cladocopium14_fst_boot.csv')[,2])
boot23<-ecdf(read.csv('/scratch/jbos/cladocopium/cladocopium23_fst_boot.csv')[,2])
boot24<-ecdf(read.csv('/scratch/jbos/cladocopium/cladocopium24_fst_boot.csv')[,2])
boot34<-ecdf(read.csv('/scratch/jbos/cladocopium/cladocopium34_fst_boot.csv')[,2])

In [14]:
#Get p value of FST Cladocopium in Acropora taxa 1 and 2 compared to null
1-boot12(fst_taxa[2,3])

[1] 0.236

In [15]:
#Get p value of FST Cladocopium in Acropora taxa 1 and 3 compared to null
1-boot13(fst_taxa[2,4])

[1] 0

In [16]:
#Get p value of FST Cladocopium in Acropora taxa 1 and 4 compared to null
1-boot14(fst_taxa[2,5])

[1] 0.291

In [17]:
#Get p value of FST Cladocopium in Acropora taxa 2 and 3 compared to null
1-boot23(fst_taxa[3,4])

[1] 0.026

In [18]:
#Get p value of FST Cladocopium in Acropora taxa 2 and 4 compared to null
1-boot24(fst_taxa[3,5])

[1] 0.013

In [19]:
#Get p value of FST Cladocopium in Acropora taxa 3 and 4 compared to null
1- boot34(fst_taxa[4,5])

[1] 0.176